# 🛠️ **OPEN PIT MINING SCHEDULE SIMULATION**

This notebook simulates a simplified open pit mining schedule using a synthetic block model. It aims to demonstrate core concepts used in mine planning and pit optimisation, including:

- Applying an economic cut-off to classify ore and waste
- Binning blocks into benches and pushbacks
- Assigning mining phases based on block value and accessibility
- Simulating period-based extraction under equipment capacity constraints
- Visualising the evolving mining sequence in 2D and 3D

### ⚙️ Key Concepts

- **Economic Cut-off**: Blocks below a given $/tonne threshold are classified as waste.
- **Benches & Pushbacks**: Mining advances downward in benches (Z) and outward in pushbacks (X/Y).
- **Scheduling**: Blocks are grouped into periods based on movement limits and ore production targets.
- **Visualisation**: Plan view, Gantt-style sequences, and 3D animations help understand mining progression.

This is a standalone scheduling approximation (not a full pit optimiser), but it illustrates key logic for phase design, sequencing, and production control.

---

## 🔍 **Step 0: Load the Block Model**

Before we begin, we load the synthetic block model file containing block locations, economic values, physical properties, and classifications (ore/waste, lithology, etc.).

The notebook supports `.csv`, `.json`, and `.parquet` formats.

We also import the necessary libraries and set the file path to access the data.


In [ ]:
# ======================================================================
# 📦 STEP 0: LOAD THE BLOCK MODEL ###
# ======================================================================

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output
import plotly.express as px
import os
from scipy.spatial import ConvexHull
from matplotlib.patches import Rectangle
import math

# --- File path setup ---
file_path = "../data/"
filename_base = "synthetic_block_model"
file_type = ".parquet"
file = os.path.join(file_path, f"{filename_base}{file_type}")
print(f"File path: {file}")

# --- Load Block Model ---

if file.endswith(".csv"):
	block_df = pd.read_csv(file)
elif file.endswith(".json"):
	block_df = pd.read_json(file)
elif file.endswith(".parquet"):
	block_df = pd.read_parquet(file)
else:
	raise ValueError("Unsupported file format")

File path: ../data/synthetic_block_model.parquet


## ⚙️ **Step 1: Define Parameters & Initial Settings**

Here we define all user-adjustable parameters related to:

- **Geometry**: Bench height and pushback width
- **Production targets**: Equipment movement limits, ore production goals
- **Economic filtering**: $/tonne cut-off for ore vs waste
- **Scheduling**: Period duration (e.g. 365 days per period)
- **Mining Phases**: Number of mining phases (e.g. 5)
- **Interactivity**: Dropdown filters for phase, block type, and depth level (Z)

These parameters drive the scheduling logic and visualisation controls used throughout the notebook.


In [ ]:
# ======================================================================
# 📦 STEP 1: DEFINE USER PARAMETERS ###
# ======================================================================

# Unit conversion
g_to_kg = 1 / 1000                      # grams to kilograms
lb_to_kg = 2.20462                      # pounds to kilograms
oz_to_g = 31.1035                       # troy ounces to grams

# --- Economic Parameters ---
mining_cost_per_tonne = 3.50            # cost AUD/t to move each tonne (ore or waste)
processing_cost_per_tonne = 12.00       # AUD/t - only applies to ore
discount_rate = 0.08                    # for optional NPV calc
exchange_rates = {
	"AUD": 1.0,
	"USD": 1 / 1.5,  					# 1 USD = 1.5 AUD
	"CAD": 0.91,
}

# --- Base commodity prices (USD) ---
gold_price_usd_per_oz = 2500  			# USD/oz
copper_price_usd_per_lb = 4.00  		# USD/lb
nsr_deduction_rate = 0.05  				# % deduction for refining costs

# --- Convert to AUD ---
gold_price_aud_per_oz = gold_price_usd_per_oz / exchange_rates["USD"]
gold_price_aud_per_g = gold_price_aud_per_oz / oz_to_g  # AUD/g
copper_price_aud_per_lb = copper_price_usd_per_lb / exchange_rates["USD"]

# --- Economic Sensitivity Parameters ---
gold_recovery = 0.90                    
copper_recovery = 0.88                  

# --- CapEx Settings ---
initial_capex = 500_000_000             # USD
sustaining_capex = 20_000_000           # Annualised or periodic (if applicable)

# --- Pit Geometry Parameters ---
ramp_gradient = 10                      # 1:10 slope
bench_height = 10                       # vertical step in metres
inter_ramp_angle_deg = 45               # Inter-Ramp Angle (IRA) in degrees
catch_berm_interval = 30                # install berm every X vertical metres
catch_berm_width = 5                    # horizontal width of catch berm in metres

# --- Buffer zone for pit shell expansion ---
buffer_margin = 100  					# metres of waste buffer added around the block model


# --- Geometry & Equipment ---
min_mining_width = 20                   # horizontal step (min pushback width) in metres
equipment_capacity_tpd = 250000         # tonnes per day moved (total material)

# --- Scheduling / Production ---
target_ore_tpd = 40000                  # tonnes per day of ore
period_days = 365                       # define mining period duration

target_ore_per_period = target_ore_tpd * period_days
target_total_movement_per_period = equipment_capacity_tpd * period_days

# 
n_phases = 5                            # number of phases in the mine plan
z_interval = 20                         # vertical interval in metres used for phase filtering

period = 1                              # starting period
moved_tonnes = 0                        # total tonnes moved by the starting period

# --- Interactive filters for phase selection ---
selected_block_type = "All"             # Options: "All", "Ore", "Waste"
selected_phase = "All"                  # Placeholder for phase filtering if applicable

# Metric selection for Gantt plot
display_metric = "tonnes"               # Options: "tonnes", "value", "volume"

# Equipment visuals (mock fleet data)
equipment_fleet = [
	{"type": "Truck", "capacity_t": 150, "count": 10},
	{"type": "Shovel", "capacity_t": 800, "count": 2}
]

# --- Mining Phases Plot Axis Defaults ---
default_xrange = (-100, 1100)
default_yrange = (0, 700)


### 🔹 **Step 1.2: Add Buffer Waste Blocks Around the Block Model**

To support realistic pit shell development, a buffer zone of waste blocks is generated around the existing block model. This allows the pit shell to expand beyond the orebody, accommodating:

- Slope constraints (inter-ramp angle, catch berms)
- Highwall formation and working space
- Ramp access development
- A more natural pit shape that does not clip exactly to the orebody boundary

The buffer:
- Extends the block model boundary by a user-defined `buffer_margin` (e.g. 100 m) in the X and Y directions
- Uses the same block dimensions and vertical resolution as the main model
- Assigns zero grade and net negative value (uneconomic waste)
- Is flagged as `"buffer"` in the optional `source` column for tracking and visualisation

These blocks are not included in the production schedule unless required for prestripping or access, but they enable slope-controlled pit shell growth and improve realism in downstream scheduling and visualisation steps.

In [ ]:
# ======================================================================
# 📦 STEP 1:ADD BUFFER WASTE BLOCKS AROUND BLOCK MODEL ###
# ======================================================================
# Buffer margin (already defined in parameters): buffer_margin

# Sort block_df by coordinates before computing spacing
block_df = block_df.sort_values(by=["x", "y", "z"]).reset_index(drop=True)

# Compute spacing (ensure spacing is positive)
dx = block_df["x"].diff().replace(0, np.nan).dropna().median()
dy = block_df["y"].diff().replace(0, np.nan).dropna().median()
dz = block_df["z"].diff().replace(0, np.nan).dropna().median()

# Safety fallback if diff returns 0
default_spacing = 10
dx = dx if dx and dx > 0 else default_spacing
dy = dy if dy and dy > 0 else default_spacing
dz = dz if dz and dz > 0 else default_spacing

print(f"✅ Block spacing: dx={dx}, dy={dy}, dz={dz}")


print("Unique X:", block_df["x"].nunique())
print("Unique Y:", block_df["y"].nunique())
print("Unique Z:", block_df["z"].nunique())

print("X range:", block_df["x"].min(), "to", block_df["x"].max())
print("Y range:", block_df["y"].min(), "to", block_df["y"].max())
print("Z range:", block_df["z"].min(), "to", block_df["z"].max())


# Safety check: ensure block size is not zero
if dx == 0 or dy == 0 or dz == 0:
    raise ValueError("❌ Detected zero block size in X, Y or Z. Check block_df and spacing assumptions.")

print(f"Block spacing: dx={dx}, dy={dy}, dz={dz}")

# Get model extents
x_min, x_max = block_df["x"].min(), block_df["x"].max()
y_min, y_max = block_df["y"].min(), block_df["y"].max()
z_vals = block_df["z"].unique()

# Limit buffer blocks to top N benches (e.g., top 2)
n_top_benches = 2
top_z_vals = sorted(block_df["z"].unique())[-n_top_benches:]

# Define extended padded range
def safe_arange(start, stop, step, label):
    count = int(np.ceil((stop - start) / step))
    if count > 5000:
        raise ValueError(f"❌ Too many values in {label} axis: {count:,}. Adjust buffer size or increase spacing.")
    return np.linspace(start, stop, count)

# Safer buffer spacing (consider 2x spacing if needed)
buffer_dx = dx * 2
buffer_dy = dy * 2

x_range_ext = safe_arange(x_min - buffer_margin, x_max + buffer_margin + buffer_dx, buffer_dx, "X")
y_range_ext = safe_arange(y_min - buffer_margin, y_max + buffer_margin + buffer_dy, buffer_dy, "Y")

print(f"Creating buffer meshgrid: X={len(x_range_ext)}, Y={len(y_range_ext)}, Z={len(top_z_vals)}")
print(f"Estimated buffer blocks: {len(x_range_ext) * len(y_range_ext) * len(top_z_vals):,}")

# Create meshgrid of all extended coordinates
xx, yy, zz = np.meshgrid(x_range_ext, y_range_ext, top_z_vals, indexing="ij")
buffer_coords = pd.DataFrame({
    "x": xx.ravel(),
    "y": yy.ravel(),
    "z": zz.ravel()
})

# Identify which blocks are *not* in the original model
original_coords = block_df[["x", "y", "z"]].drop_duplicates()
buffer_coords = buffer_coords.merge(original_coords, how="left", on=["x", "y", "z"], indicator=True)
buffer_coords = buffer_coords[buffer_coords["_merge"] == "left_only"].drop(columns=["_merge"])

# Assign buffer block attributes
buffer_coords["class"] = "Waste"
buffer_coords["Au_grade"] = 0.0
buffer_coords["Cu_grade"] = 0.0
buffer_coords["density"] = block_df["density"].mean()
buffer_coords["tonnes"] = buffer_coords["density"] * dx * dy * dz
buffer_coords["volume_m3"] = dx * dy * dz
buffer_coords["value_Au_per_tonne"] = 0.0
buffer_coords["value_Cu_per_tonne"] = 0.0
buffer_coords["value_per_tonne"] = 0.0
buffer_coords["net_value"] = -mining_cost_per_tonne  # Always uneconomic
buffer_coords["source"] = "buffer"  # Optional: useful to distinguish them

# Align columns with block_df
for col in block_df.columns:
    if col not in buffer_coords.columns:
        buffer_coords[col] = np.nan
buffer_coords = buffer_coords[block_df.columns]  # Ensure column order

# Append to block_df
block_df = pd.concat([block_df, buffer_coords], ignore_index=True)


✅ Block spacing: dx=25.0, dy=25.0, dz=10.0
Unique X: 144
Unique Y: 96
Unique Z: 72
X range: 12.5 to 3587.5
Y range: 12.5 to 2387.5
Z range: 5.0 to 715.0
Block spacing: dx=25.0, dy=25.0, dz=10.0
Creating buffer meshgrid: X=77, Y=53, Z=2
Estimated buffer blocks: 8,162


## 🧮 Step 2: Pit Shell, Phases, Prestrip & Scheduling
In this step, we simulate the progressive growth of a realistic open pit mine using geotechnical and operational constraints. The logic mimics how pit shells expand in the real world, factoring in:

🔍 What We Do in This Step:

**1. Define a Central Growth Point** \
The pit expands concentrically from the centre of the orebody.

**2. Apply Geotechnical Constraints** \
Based on the Inter-Ramp Angle (IRA) and Catch Berms, we calculate the maximum allowable lateral extent (radius) for each block by depth.

**3. Determine Shell Inclusion** \
Blocks are flagged as *within the pit shell* if they lie inside the allowable slope envelope.

**4. Apply Economic Filter** \
Only blocks with a value above the cut-off and within the pit geometry are selected.

**5. Assign Mining Phases by Value** \
Economic blocks are sorted and divided into mining phases (e.g., pre-strip, Phase 1, 2, etc.).

**6. Group into Pushbacks & Benches** \
Blocks are grouped by bench height and minimum mining width to enable equipment access and scheduling.

**7. Assign Periods Based on Capacity** \
Material is scheduled period-by-period, obeying the equipment capacity (tonnes moved per period).

**8. Build a Summary Table** \
We calculate total ore, waste, movement, profit, and discounted NPV per period to evaluate the viability of the schedule.

In [ ]:
# ======================================================================
# 📦 STEP 2: PIT SHELL, PHASES, PRESTRIL & SCHEDULING ###
# ======================================================================

# --- 1. Economic Value Calculation ---
block_df["processing_cost"] = np.where(
	block_df["class"] == "Ore", processing_cost_per_tonne, 0
)
block_df["value_Au_per_tonne"] = (
	block_df["Au_grade"] * gold_price_aud_per_g * gold_recovery
)
block_df["value_Cu_per_tonne"] = (
	block_df["Cu_grade"] / 100 * 1000 * lb_to_kg * copper_price_aud_per_lb * copper_recovery
)
block_df["value_per_tonne"] = (
	block_df["value_Au_per_tonne"] + block_df["value_Cu_per_tonne"]
)
block_df["value_per_tonne"] *= (1 - nsr_deduction_rate)
block_df["net_value"] = (
	block_df["value_per_tonne"] - mining_cost_per_tonne - block_df["processing_cost"]
)

# --- 2. Start with All Blocks ---
all_blocks = block_df.copy()
all_blocks["block_type"] = np.where(all_blocks["class"] == "Ore", "Ore", "Waste")

# --- 3. Pit Shell Geometry ---
x_centre = all_blocks["x"].mean()
y_centre = all_blocks["y"].mean()
max_z = all_blocks["z"].max()
ira_rad = math.radians(inter_ramp_angle_deg)
slope_ratio = math.tan(ira_rad)

all_blocks["depth"] = max_z - all_blocks["z"]
all_blocks["num_berms"] = (all_blocks["depth"] // catch_berm_interval).astype(int)
all_blocks["berm_offset"] = all_blocks["num_berms"] * catch_berm_width

# Elliptical shaping of pit
# Padding the pit shell envelope beyond the orebody
x_padding = 100  							# meters
y_padding = 100

x_range = (all_blocks["x"].max() - all_blocks["x"].min()) + 2 * x_padding
y_range = (all_blocks["y"].max() - all_blocks["y"].min()) + 2 * y_padding

x_ratio = 1.5  								# Stretch pit more in X (elongated shape)
y_ratio = 1.0  								# Keep Y as is

all_blocks["x_factor"] = (all_blocks["x"] - x_centre) / (x_range / x_ratio)
all_blocks["y_factor"] = (all_blocks["y"] - y_centre) / (y_range / y_ratio)
all_blocks["elliptical_distance"] = np.sqrt(
	all_blocks["x_factor"]**2 + all_blocks["y_factor"]**2
	)

# Use same slope ratio and berm offset
all_blocks["allowed_radius"] = all_blocks["depth"] * slope_ratio
all_blocks["effective_radius"] = all_blocks["allowed_radius"] - all_blocks["berm_offset"]
all_blocks["within_shell"] = all_blocks["elliptical_distance"] <= all_blocks["effective_radius"]

all_blocks["has_access"] = all_blocks["elliptical_distance"] <= all_blocks["depth"] * ramp_gradient
all_blocks["is_economic"] = all_blocks["net_value"] >= 0

# --- 4. Filter Economic & Accessible Blocks for Scheduling ---
scheduled_blocks = all_blocks[
	all_blocks["within_shell"] & all_blocks["has_access"] & all_blocks["is_economic"]
].copy()
scheduled_blocks["phase"] = None

# --- 5. Add Phase 0: Top Bench Waste (Prestrip) ---
early_benches = scheduled_blocks["z"].unique()
top_benches = sorted(early_benches)[-2:]  # Top 2 benches

prestrip_blocks = all_blocks[
	(all_blocks["within_shell"]) &
	(all_blocks["has_access"]) &
	(~all_blocks["is_economic"]) &
	(all_blocks["z"].isin(top_benches))
].copy()

if not prestrip_blocks.empty:
	prestrip_blocks["phase"] = "Phase 0"
	prestrip_blocks["period"] = 0

	# Ensure all columns match
	for col in scheduled_blocks.columns:
		if col not in prestrip_blocks.columns:
			prestrip_blocks[col] = np.nan

	scheduled_blocks = pd.concat(
		[prestrip_blocks[scheduled_blocks.columns], scheduled_blocks], ignore_index=True
		)

# --- 6. Bench & Pushback Binning ---
scheduled_blocks["bench"] = (scheduled_blocks["z"] // bench_height).astype(int) * bench_height
scheduled_blocks["pushback_x"] = (
	scheduled_blocks["x"] // min_mining_width).astype(int
	) * min_mining_width
scheduled_blocks["pushback_y"] = (
	scheduled_blocks["y"] // min_mining_width
	).astype(int) * min_mining_width

# --- 7. Assign Mining Phases (excluding Phase 0) ---
to_phase = scheduled_blocks[~scheduled_blocks["phase"].notnull()]
to_phase = to_phase.sort_values(by="net_value", ascending=False).reset_index(drop=True)
to_phase["phase"] = pd.qcut(
	to_phase.index, q=n_phases, labels=[f"Phase {i+1}" for i in range(n_phases)]
)
scheduled_blocks.update(to_phase)

# --- 8. Extraction Order & Scheduling ---
bin_groups = scheduled_blocks.groupby(["bench", "pushback_y", "pushback_x"])
extraction_order = (
	bin_groups["value_per_tonne"].mean()
	.reset_index()
	.sort_values(by=["bench", "pushback_y", "pushback_x"], ascending=[False, True, True])
	.reset_index(drop=True)
)

scheduled_blocks["period"] = scheduled_blocks.get("period", -1)
period = 1
moved_tonnes = 0

for idx, row in extraction_order.iterrows():
	mask = (
		(scheduled_blocks["bench"] == row["bench"]) &
		(scheduled_blocks["pushback_y"] == row["pushback_y"]) &
		(scheduled_blocks["pushback_x"] == row["pushback_x"]) &
		(scheduled_blocks["period"] == -1)
	)
	block_tonnes = scheduled_blocks[mask]["tonnes"].sum()
	if moved_tonnes + block_tonnes > target_total_movement_per_period:
		period += 1
		moved_tonnes = 0
	scheduled_blocks.loc[mask, "period"] = period
	moved_tonnes += block_tonnes

# --- 9. Summary Table ---
summary = scheduled_blocks.groupby(["period", "block_type"])["tonnes"].sum().unstack(fill_value=0)
summary["total_moved"] = summary.sum(axis=1)
summary["cumulative_movement"] = summary["total_moved"].cumsum()
summary["met_capacity"] = summary["total_moved"] <= target_total_movement_per_period
summary.reset_index(inplace=True)

# --- 10. Economic Metrics & Strip Ratio ---
summary["ore_tonnes"] = scheduled_blocks[scheduled_blocks["block_type"] == "Ore"].groupby("period")["tonnes"].sum().reindex(summary["period"], fill_value=0)
summary["waste_tonnes"] = summary.get("Waste", 0)
summary["strip_ratio"] = np.where(
	summary["ore_tonnes"] > 0,
	summary["waste_tonnes"] / summary["ore_tonnes"],
	np.nan
)

# --- 11. NPV & Payback ---
revenue = scheduled_blocks.groupby("period")["value_per_tonne"].sum().reindex(summary["period"], fill_value=0)
mining_cost = scheduled_blocks.groupby("period")["tonnes"].sum().reindex(summary["period"], fill_value=0) * mining_cost_per_tonne
processing_cost = (
	scheduled_blocks[scheduled_blocks["block_type"] == "Ore"]
	.groupby("period")["tonnes"].sum().reindex(summary["period"], fill_value=0) * processing_cost_per_tonne
)

summary["revenue"] = revenue
summary["mining_cost"] = mining_cost
summary["processing_cost"] = processing_cost
summary["profit"] = summary["revenue"] - summary["mining_cost"] - summary["processing_cost"]
summary["discount_factor"] = 1 / (1 + discount_rate) ** summary["period"]
summary["discounted_profit"] = summary["profit"] * summary["discount_factor"]
summary["cumulative_npv"] = summary["discounted_profit"].cumsum()
summary["cumulative_cashflow"] = summary["cumulative_npv"] - initial_capex
summary["payback_met"] = summary["cumulative_cashflow"] >= 0

payback_period = summary[summary["payback_met"]].head(1)["period"].values[0] if summary["payback_met"].any() else None

In [ ]:
#### FOR DEBUGGING ONLY ####
print("Ore blocks scheduled:", len(scheduled_blocks[scheduled_blocks["block_type"] == "Ore"]))
print("Net value summary:\n", block_df["net_value"].describe())
print("Economic blocks:", (block_df["net_value"] >= 0).sum())
valid_blocks = all_blocks[all_blocks["within_shell"] & all_blocks["has_access"]]
print("Within shell and accessible:", len(valid_blocks))
print("Of those, economic blocks:", len(valid_blocks[valid_blocks["net_value"] >= 0]))

print(block_df[["Au_grade", "Cu_grade", "value_per_tonne", "net_value"]].describe())


Ore blocks scheduled: 22146
Net value summary:
 count    28616.000000
mean       259.419782
std         55.876420
min         -3.500000
25%        242.004383
50%        266.463359
75%        289.412922
max        352.174349
Name: net_value, dtype: float64
Economic blocks: 27716
Within shell and accessible: 27100
Of those, economic blocks: 26650
           Au_grade      Cu_grade  value_per_tonne     net_value
count  28616.000000  28616.000000     28616.000000  28616.000000
mean       1.922793      0.670311       272.333257    259.419782
std        0.466088      0.146235        59.030633     55.876420
min        0.000000      0.000000         0.000000     -3.500000
25%        1.738356      0.635503       254.273490    242.004383
50%        1.991498      0.683745       281.963359    266.463359
75%        2.207058      0.744143       304.912922    289.412922
max        2.752862      0.956505       367.674349    352.174349


In [ ]:
scheduled_blocks.columns

Index(['x', 'y', 'z', 'Au_grade', 'Cu_grade', 'density', 'mag_sus',
       'resistivity', 'gamma', 'UCS', 'RMR', 'Axb', 'BWi', 'DWi',
       'grindability', 'floatability_index', 'reagent_demand', 'lith_code',
       'stratigraphy', 'alteration', 'grain_size', 'oxidation_zone',
       'processing_route', 'currency', 'value_Au_per_tonne',
       'value_Cu_per_tonne', 'value_per_tonne', 'class', 'domain', 'volume_m3',
       'tonnes', 'block_id', 'value_smu', 'class_smu', 'processing_cost',
       'net_value', 'block_type', 'depth', 'num_berms', 'berm_offset',
       'x_factor', 'y_factor', 'elliptical_distance', 'allowed_radius',
       'effective_radius', 'within_shell', 'has_access', 'is_economic',
       'phase', 'bench', 'pushback_x', 'pushback_y', 'period'],
      dtype='object')

## 📊 Step 3: Visualise the Mining Phases

### ⛏️ Plan View with Phase Colours (Interactive)
- Select Z-slice using a dropdown
- View horizontal mining slices coloured by phase
- Future feature: add pit shell polygons and toggle individual phases on/off

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

# Define dropdown Z levels from scheduled_blocks
z_min, z_max = scheduled_blocks["z"].min(), scheduled_blocks["z"].max()
z_levels = np.arange(z_min, z_max + z_interval, z_interval)

# Extract unique phases for filtering
unique_phases = sorted(scheduled_blocks["phase"].dropna().unique().tolist())

# --- Widgets ---
z_dropdown = widgets.Dropdown(
	options=[(f"{int(z)} m", z) for z in z_levels],
	value=z_levels[len(z_levels) // 2],
	description="Z Level:",
	layout=widgets.Layout(width='300px')
)

phase_multiselect = widgets.SelectMultiple(
	options=unique_phases,
	value=tuple(unique_phases),
	description="Phases:",
	layout=widgets.Layout(width='300px')
)

metric_dropdown = widgets.Dropdown(
	options=["tonnes", "value", "volume"],
	value="tonnes",
	description="Display Metric:",
	layout=widgets.Layout(width='300px')
)

equip_mode_toggle = widgets.ToggleButtons(
	options=["Bench", "Period"],
	value="Bench",
	description="Equipment View:",
	layout=widgets.Layout(width='300px')
)

# --- Main Plot Function ---
def plot_interactive(z_value, selected_phases, metric, equip_mode):
	clear_output(wait=True)
	display(widgets.HBox([z_dropdown, phase_multiselect, metric_dropdown, equip_mode_toggle]))

	slice_df = scheduled_blocks[np.isclose(scheduled_blocks["z"], z_value)]
	if selected_phases:
		slice_df = slice_df[slice_df["phase"].isin(selected_phases)]

	if slice_df.empty:
		print(f"No data at Z={z_value}")
		return

	# --- Size scaling by metric ---
	if metric == "tonnes":
		size_vals = slice_df["tonnes"]
	elif metric == "value":
		size_vals = slice_df["value_per_tonne"]
	elif metric == "volume":
		size_vals = slice_df["volume_m3"]

	# --- Main Scatter ---
	fig = px.scatter(
		slice_df,
		x="x", y="y",
		color="phase",
		size=size_vals,
		size_max=20,
		title=f"Mining Phases at Z = {z_value:.1f} m",
		hover_data={
			"x": False,
			"y": False,
			"phase": True,
			"tonnes": ":,.0f",
			"value_per_tonne": ":,.0f",
			"volume_m3": ":,.0f",
			"period": True
		},
		labels={
			"value_per_tonne": "Value $/t",
			"tonnes": "Tonnes",
			"volume_m3": "Volume (m³)",
			"phase": "Phase",
			"period": "Period"
		}
	)

	# --- Equipment Visuals Overlay ---
	equipment_shapes = []
	label_positions = []

	if equip_mode == "Bench":
		base_x = default_xrange[0] + 40
		base_y = default_yrange[1] - 80  # move visuals down slightly

		for eq in equipment_fleet:
			if eq["type"] == "Shovel":
				equipment_shapes.append(dict(
					type="rect",
					x0=base_x,
					x1=base_x + 30,
					y0=base_y,
					y1=base_y + 10,
					line=dict(color="black"),
					fillcolor="green"
				))
				label_positions.append((base_x, base_y + 12, "Shovel"))

			elif eq["type"] == "Truck":
				for i in range(eq["count"]):
					truck_x = base_x + (i % 5) * 22
					truck_y = base_y - 40 - (i // 5) * 20
					equipment_shapes.append(dict(
						type="rect",
						x0=truck_x,
						x1=truck_x + 15,
						y0=truck_y,
						y1=truck_y + 8,
						line=dict(color="black"),
						fillcolor="gray"
					))
					if i == 0:
						label_positions.append((truck_x, truck_y - 5, "Truck"))

	elif equip_mode == "Period":
		active_period = scheduled_blocks[scheduled_blocks["z"] == z_value]["period"].mode()[0]
		active_blocks = scheduled_blocks[scheduled_blocks["period"] == active_period]
		if not active_blocks.empty:
			for eq in equipment_fleet:
				if eq["type"] == "Shovel":
					equipment_shapes.append(dict(
						type="rect",
						x0=active_blocks["x"].mean() - 10,
						x1=active_blocks["x"].mean() + 10,
						y0=active_blocks["y"].max() - 10,
						y1=active_blocks["y"].max(),
						line=dict(color="black"),
						fillcolor="green"
					))
					label_positions.append((active_blocks["x"].mean() - 10, active_blocks["y"].max() - 15, "Shovel"))

	fig.update_layout(
		xaxis_title="X (m)",
		yaxis_title="Y (m)",
		xaxis=dict(range=default_xrange),
		yaxis=dict(range=default_yrange, scaleanchor="x", scaleratio=1),
		height=700,
		legend_title="Phase",
		margin=dict(l=50, r=250, t=50, b=50)
	)

	# Add optional shell outline
	shell_blocks = all_blocks[(all_blocks["within_shell"]) & (np.isclose(all_blocks["z"], z_value))]

	if not shell_blocks.empty:
		fig.add_trace(go.Scatter(
			x=shell_blocks["x"],
			y=shell_blocks["y"],
			mode="markers",
			marker=dict(color="lightgray", size=4, symbol="square"),
			name="Pit Shell",
			hoverinfo="skip",
			showlegend=True
		))

		
	# --- Add Equipment Shapes ---
	fig.update_layout(shapes=equipment_shapes)
	for x, y, label in label_positions:
		fig.add_annotation(x=x, y=y, text=label, showarrow=False)

	fig.show()

# --- Observers ---
def on_filter_change(change):
	plot_interactive(
		z_dropdown.value,
		list(phase_multiselect.value),
		metric_dropdown.value,
		equip_mode_toggle.value
	)

z_dropdown.observe(on_filter_change, names="value")
phase_multiselect.observe(on_filter_change, names="value")
metric_dropdown.observe(on_filter_change, names="value")
equip_mode_toggle.observe(on_filter_change, names="value")

# --- Initial Display ---
plot_interactive(
	z_dropdown.value,
	list(phase_multiselect.value),
	metric_dropdown.value,
	equip_mode_toggle.value
)


## Step 4: Visualise Mining Schedule & Sequence
### 📆 Gantt-style Strip Plot
- Each dot is a bench extracted in a period
- Interact with dropdowns to highlight block types (ore/waste) and target periods
- Shows scheduling progression from top-down

### 📦 3D Mining Animation
- Animated 3D scatter plot of blocks
- Colour-coded by period of extraction
- Useful to visualise pit deepening and lateral expansion over time

In [ ]:
metric_dropdown = widgets.Dropdown(
	options=["tonnes", "value", "volume"],
	value="tonnes",
	description="Display Metric:"
)

block_type_dropdown = widgets.Dropdown(
	options=["All", "Ore", "Waste"],
	value="All",
	description="Block Type:"
)

highlight_period_slider = widgets.IntSlider(
	value=1,
	min=1,
	max=scheduled_blocks["period"].max(),
	step=1,
	description="Highlight Period:"
)

def update_gantt_stack(metric, block_type, highlight_period):
	clear_output(wait=True)

	df = scheduled_blocks.copy()
	if block_type != "All":
		df = df[df["block_type"] == block_type]

	df["highlight"] = df["period"] == highlight_period
	df["period_str"] = df["period"].astype(str)  # Treat as categorical

	plt.figure(figsize=(16, 6))
	sns.stripplot(
		data=df,
		x="period_str", y="bench",
		hue="highlight",
		palette={True: "red", False: "lightgrey"},
		jitter=0.3, alpha=0.6, dodge=False
	)
	plt.gca().invert_yaxis()
	plt.xticks(rotation=90)
	plt.locator_params(axis="x", nbins=20)
	plt.xlabel("Period")
	plt.ylabel("Bench (Z)")
	plt.title("Gantt-style Mining Sequence with Highlighted Period")
	plt.xticks(rotation=90)
	plt.legend(title="Highlighted", loc='upper left')
	plt.grid(True)
	plt.tight_layout()
	plt.show()

	display(metric_dropdown, block_type_dropdown, highlight_period_slider)


widgets.interact(
	update_gantt_stack,
	metric=metric_dropdown,
	block_type=block_type_dropdown,
	highlight_period=highlight_period_slider
)

# --- 3D Animation Plot ---
fig = px.scatter_3d(
	scheduled_blocks,
	x="x", y="y", z="z",
	color="period",
	animation_frame="period",
	size_max=4,
	color_continuous_scale="Viridis",
	opacity=0.5,
	title="Animated 3D Mining Schedule"
)
x_range = [scheduled_blocks["x"].min(), scheduled_blocks["x"].max()]
y_range = [scheduled_blocks["y"].min(), scheduled_blocks["y"].max()]
z_range = [scheduled_blocks["z"].min(), scheduled_blocks["z"].max()]

fig.update_layout(
	scene=dict(
		xaxis=dict(title='X', range=x_range),
		yaxis=dict(title='Y', range=y_range),
		zaxis=dict(title='Z', range=z_range),
	)
)

fig.show()

interactive(children=(Dropdown(description='Display Metric:', options=('tonnes', 'value', 'volume'), value='to…

In [ ]:


# --- Enhanced Gantt-style Visualisation (Global + Highlighted Period) ---

# Widgets
block_type_dropdown = widgets.Dropdown(
	options=["All", "Ore", "Waste"],
	value="All",
	description="Block Type:"
)

highlight_period_slider = widgets.IntSlider(
	value=1,
	min=1,
	max=scheduled_blocks["period"].max(),
	step=1,
	description="Highlight Period:"
)

def update_gantt_all(block_type, highlight_period):
	clear_output(wait=True)

	# Filter
	plot_df = scheduled_blocks.copy()
	if block_type != "All":
		plot_df = plot_df[plot_df["block_type"] == block_type]

	# Highlight current period
	plot_df["highlight"] = plot_df["period"] == highlight_period

	plt.figure(figsize=(14, 6))
	sns.stripplot(
		data=plot_df, x="period", y="bench",
		hue="highlight", palette={True: "red", False: "lightgrey"},
		jitter=0.3, alpha=0.6, dodge=False
	)
	plt.gca().invert_yaxis()
	plt.xlabel("Period")
	plt.ylabel("Bench (Z)")
	plt.title("Full Gantt-style Mining Sequence (All Periods) – Highlighted Period in Red")
	plt.legend(title="Highlighted")
	plt.grid(True)
	plt.tight_layout()
	plt.show()

	display(block_type_dropdown, highlight_period_slider)

widgets.interact(update_gantt_all,
				 block_type=block_type_dropdown,
				 highlight_period=highlight_period_slider)

# --- 3D Animation with Plotly ---

fig = px.scatter_3d(
	scheduled_blocks,
	x="x", y="y", z="z",
	color="period",
	animation_frame="period",
	size_max=4,
	color_continuous_scale="Viridis",
	opacity=0.5,
	title="Animated 3D Mining Schedule"
)
fig.update_layout(scene=dict(xaxis_title='X', yaxis_title='Y', zaxis_title='Z'))
fig.show()

interactive(children=(Dropdown(description='Block Type:', options=('All', 'Ore', 'Waste'), value='All'), IntSl…

### STEP x: Export to CSV

In [ ]:
scheduled_blocks.to_csv("../data/scheduled_block_model.csv", index=False)
summary.to_csv("../data/schedule_summary.csv", index=False)
print("✅ Exported pit schedule and summary to ../data/")

In [ ]:
scheduled_blocks.head()

In [ ]:
# --- STEP 1: User-Defined Parameters for Pit Shell & Scheduling ---

# Geometry & Equipment
bench_height = 10  # vertical step in metres
min_mining_width = 20  # horizontal step (min pushback width) in metres
equipment_capacity_tpd = 100000  # tonnes/day total material moved

# Economic Filter
economic_cutoff = 20  # $/tonne cutoff for ore

# Scheduling / Production
target_ore_tpd = 40000
period_days = 30
target_ore_per_period = target_ore_tpd * period_days
target_total_movement_per_period = equipment_capacity_tpd * period_days

# Interactive filters
selected_block_type = "All"  # Options: "All", "Ore", "Waste"
selected_phase = "All"        # Placeholder for phase filtering if applicable

# Metric selection for Gantt plot
display_metric = "tonnes"  # Options: "tonnes", "value", "volume"

# Equipment visuals (mock fleet data)
equipment_fleet = [
	{"type": "Truck", "capacity_t": 150, "count": 10},
	{"type": "Shovel", "capacity_t": 800, "count": 2}
]

# --- STEP 2: Filter Economic Ore ---
block_df["block_type"] = np.where(
	block_df["value_per_tonne"] >= economic_cutoff, "Ore", "Waste"
)
scheduled_blocks = block_df.copy()

# --- STEP 3: Bin Z to Bench Levels ---
scheduled_blocks["bench"] = (scheduled_blocks["z"] // bench_height).astype(int) * bench_height

# --- STEP 4: Assign Pushback X/Y Bins ---
scheduled_blocks["pushback_x"] = (scheduled_blocks["x"] // min_mining_width).astype(int) * min_mining_width
scheduled_blocks["pushback_y"] = (scheduled_blocks["y"] // min_mining_width).astype(int) * min_mining_width

# --- STEP 5: Rank Bins and Define Extraction Order ---
bin_groups = scheduled_blocks.groupby(["bench", "pushback_y", "pushback_x"])
extraction_order = (
	bin_groups["value_per_tonne"].mean()
	.reset_index()
	.sort_values(by=["bench", "pushback_y", "pushback_x"], ascending=[False, True, True])
	.reset_index(drop=True)
)

# Assign mining period to each bin group
scheduled_blocks["period"] = -1
period = 1
moved_tonnes = 0
for idx, row in extraction_order.iterrows():
	mask = (
		(scheduled_blocks["bench"] == row["bench"]) &
		(scheduled_blocks["pushback_y"] == row["pushback_y"]) &
		(scheduled_blocks["pushback_x"] == row["pushback_x"])
	)
	block_tonnes = scheduled_blocks[mask]["tonnes"].sum()
	if moved_tonnes + block_tonnes > target_total_movement_per_period:
		period += 1
		moved_tonnes = 0
	scheduled_blocks.loc[mask, "period"] = period
	moved_tonnes += block_tonnes

# --- STEP 6: Summary Table ---
summary = scheduled_blocks.groupby(["period", "block_type"])["tonnes"].sum().unstack(fill_value=0)
summary["total_moved"] = summary.sum(axis=1)
summary["cumulative_movement"] = summary["total_moved"].cumsum()
summary["met_capacity"] = summary["total_moved"] <= target_total_movement_per_period
summary.reset_index(inplace=True)

# --- STEP 7: Updated Gantt & Visualisations ---
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display, clear_output

metric_dropdown = widgets.Dropdown(
	options=["tonnes", "value", "volume"],
	value="tonnes",
	description="Display Metric:"
)

block_type_dropdown = widgets.Dropdown(
	options=["All", "Ore", "Waste"],
	value="All",
	description="Block Type:"
)

highlight_period_slider = widgets.IntSlider(
	value=1,
	min=1,
	max=scheduled_blocks["period"].max(),
	step=1,
	description="Highlight Period:"
)

def update_gantt_stack(metric, block_type, highlight_period):
	clear_output(wait=True)

	df = scheduled_blocks.copy()
	if block_type != "All":
		df = df[df["block_type"] == block_type]

	df["highlight"] = df["period"] == highlight_period
	df["period_str"] = df["period"].astype(str)  # Treat as categorical

	plt.figure(figsize=(16, 6))
	sns.stripplot(
		data=df,
		x="period_str", y="bench",
		hue="highlight",
		palette={True: "red", False: "lightgrey"},
		jitter=0.3, alpha=0.6, dodge=False
	)
	plt.gca().invert_yaxis()
	plt.xlabel("Period")
	plt.ylabel("Bench (Z)")
	plt.title("Gantt-style Mining Sequence with Highlighted Period")
	plt.xticks(rotation=90)
	plt.legend(title="Highlighted", loc='upper left')
	plt.grid(True)
	plt.tight_layout()
	plt.show()

	display(metric_dropdown, block_type_dropdown, highlight_period_slider)

widgets.interact(
	update_gantt_stack,
	metric=metric_dropdown,
	block_type=block_type_dropdown,
	highlight_period=highlight_period_slider
)

# --- 3D Animation Plot ---
fig = px.scatter_3d(
	scheduled_blocks,
	x="x", y="y", z="z",
	color="period",
	animation_frame="period",
	size_max=4,
	color_continuous_scale="Viridis",
	opacity=0.5,
	title="Animated 3D Mining Schedule"
)
fig.update_layout(
	scene=dict(
		xaxis_title='X',
		yaxis_title='Y',
		zaxis_title='Z',
		aspectmode='data'  # Fixes scaling to match data ranges
	)
)

fig.show()


🔢 1. Equipment Visual Overlay (Plan View)

In [ ]:
# --- Equipment Overlay on Plan View ---

def overlay_equipment(ax, positions, shape="rectangle", label="Truck", size=20):
	for (x, y) in positions:
		if shape == "rectangle":
			rect = plt.Rectangle((x - size / 2, y - size / 2), size, size,
								 linewidth=1, edgecolor="black", facecolor="grey", alpha=0.6)
			ax.add_patch(rect)
			ax.text(x, y, label, ha='center', va='center', fontsize=8, color='white')

equipment_positions = [(50, 50), (100, 100), (150, 50)]  # Example XY positions

def plot_z_slice(z_value):
	clear_output(wait=True)

	slice_df = scheduled_blocks[np.isclose(scheduled_blocks["z"], z_value)]
	if slice_df.empty:
		print(f"No data at Z={z_value}")
		display(z_dropdown)
		return

	fig, ax = plt.subplots(figsize=(10, 8))
	sns.scatterplot(
		data=slice_df,
		x="x", y="y", hue="phase",
		palette="Set1", s=80, edgecolor="black", ax=ax
	)

	overlay_equipment(ax, equipment_positions, label="Truck", size=20)

	plt.title(f"Mining Phases at Z={z_value:.1f} m")
	plt.xlabel("X (m)")
	plt.ylabel("Y (m)")
	plt.gca().set_aspect("equal")
	plt.legend(title="Phase", bbox_to_anchor=(1.05, 1), loc='upper left')
	plt.grid(True)
	plt.tight_layout()
	plt.show()
	display(z_dropdown)

def on_filter_change(change):
	plot_z_slice(z_dropdown.value, list(phase_multiselect.value))

z_dropdown.observe(on_filter_change, names="value")
phase_multiselect.observe(on_filter_change, names="value")

# --- Toggle for Metric (tonnes / value / volume) ---
metric_toggle = widgets.Dropdown(
	options=["tonnes", "value_per_tonne", "volume_m3"],
	value="tonnes",
	description="Metric:"
)

# --- Initial Display ---
display(z_dropdown, phase_multiselect)
plot_z_slice(z_dropdown.value)

def get_metric_value(df, metric):
	if metric == "tonnes":
		return df["tonnes"]
	elif metric == "value_per_tonne":
		return df["value_per_tonne"] * df["tonnes"]
	elif metric == "volume_m3":
		return df["volume_m3"]

